# 01 - Batch Source to Bronze Delta

Generate a visible CSV source, upload it to local MinIO, read it with PySpark, and append it to the raw bronze Delta table.

In [ ]:
import sys
from dataclasses import asdict
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None

from insightsstream.delta_pipeline import DeltaSettings, create_delta_spark_session, delta_history, ingest_batch_to_bronze, reset_lake, upload_source
from producer.generate_source import generate_events

settings = DeltaSettings.from_env()
spark = create_delta_spark_session("InsightsStream-Batch-Delta", settings)

In [ ]:
RESET_LAKE = True
if RESET_LAKE:
    print(f"Removed {reset_lake(settings)} previous MinIO objects")

source_path = ROOT / "data" / "source" / "generated_events.csv"
generate_events(source_path, row_count=1000, seed=42, invalid_rate=0.01, duplicate_rate=0.02)
source_uri = upload_source(source_path, settings)
print(f"Host CSV: {source_path}")
print(f"MinIO source: {source_uri}")

In [ ]:
source = spark.read.option("header", True).csv(source_uri)
source.show(10, truncate=False)

run = ingest_batch_to_bronze(spark, source_uri, settings)
print(asdict(run))

bronze = spark.read.format("delta").load(settings.uri("bronze/events"))
bronze.createOrReplaceTempView("bronze_events")
spark.sql("SELECT source_type, metric, COUNT(*) AS rows FROM bronze_events GROUP BY source_type, metric ORDER BY rows DESC").show()
delta_history(spark, "bronze/events", settings).select("version", "timestamp", "operation").show(truncate=False)